### ASTRADB VectorStore
Go from app idea to production with the AI Platform with Astra DB, the ultra-low latency database made for AI and Langflow, the low-code RAG IDE
https://www.datastax.com/

In [1]:
!pip install \
    "langchain>=0.3.23,<0.4" \
    "langchain-core>=0.3.52,<0.4" \
    "langchain-astradb>=0.6,<0.7"

No global/local python version has been set yet. Please set the global/local version by typing:
pyenv global 3.7.4
pyenv local 3.7.4


In [ ]:
ASTRA_DB_API_ENDPOINT="####"
ASTRA_DB_APPLICATION_TOKEN="####"

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name ="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2640.46it/s]


In [10]:
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [11]:
!uv add langchain-astradb

Resolved 189 packages in 21ms
Checked 162 packages in 68ms


In [12]:
from langchain_astradb import AstraDBVectorStore
vector_store = AstraDBVectorStore(
    embedding=embeddings,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    collection_name="astra_vector_langchain",
    token= ASTRA_DB_APPLICATION_TOKEN,
    namespace=None
)
vector_store

In [13]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [15]:
vector_store.add_documents(documents=documents)

['bd4640bd55b247e8a779ed159e855e36',
 'eecabea872544e0a9e470ebd69d83471',
 '5c7a43430c16406a960c0b36a0fd4a31',
 '94a0c8511cb040cdbd4a752cf954a0ab',
 '2cdc54cc121b48aa8ee5aaac5d1e8ebf',
 '991a795b31ce4a8f9c2509656433fef1',
 'c9b72e3099424fe4ad47bf069901f057',
 'd92db878dd7b4c9ea6b494c876d30605',
 '8468a5c27a374113ade9ebb7c2158550',
 'bab07f94c1d64175b6a52c2ae082dae8']

In [16]:
## Search from vector store DB

vector_store.similarity_search("what is the weather")

[Document(id='eecabea872544e0a9e470ebd69d83471', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='8468a5c27a374113ade9ebb7c2158550', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='d92db878dd7b4c9ea6b494c876d30605', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='c9b72e3099424fe4ad47bf069901f057', metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.')]

In [18]:
results = vector_store.similarity_search(
    "Langchain provides abstractions to make working with LLms easy",
    k=3,
    filter={"source": "tweet"},

)
for res in results:
    print(f'* "{res.page_content}", metadata={res.metadata}')

* "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* "LangGraph is the best framework for building stateful, agentic applications!", metadata={'source': 'tweet'}
* "I have a bad feeling I am going to get deleted :(", metadata={'source': 'tweet'}


In [19]:
results = vector_store.similarity_search_with_score(
    "LangChain provides abstractions to make working with LLMs easy",
    k=3,
    filter={"source": "tweet"},
)
for res, score in results:
    print(f'* [SIM= {score:.2f}] "{res.page_content}", metadata= {res.metadata}')

* [SIM= 0.79] "Building an exciting new project with LangChain - come check it out!", metadata= {'source': 'tweet'}
* [SIM= 0.75] "LangGraph is the best framework for building stateful, agentic applications!", metadata= {'source': 'tweet'}
* [SIM= 0.51] "I have a bad feeling I am going to get deleted :(", metadata= {'source': 'tweet'}


In [21]:
## Retriever 
retriever=vector_store.as_retriever(
  search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.5},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='94a0c8511cb040cdbd4a752cf954a0ab', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

In [22]:
### Retriever
retriever=vector_store.as_retriever(
  search_type="mmr",
    search_kwargs={"k": 1},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='94a0c8511cb040cdbd4a752cf954a0ab', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

### Build RAG Chain With LCEL 

In [23]:
## LLM GQOK
from langchain_groq import ChatGroq
import os

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm= ChatGroq(
    model_name = "openai/gpt-oss-120b"
)

In [24]:
llm.invoke("hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond. Probably greeting. No special instructions. Just friendly.'}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 72, 'total_tokens': 112, 'completion_time': 0.083043448, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.016072804, 'prompt_tokens_details': None, 'queue_time': 0.216907719, 'total_time': 0.099116252}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_27194a498a', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12013-26b5-7481-bf2c-49efa165c7a9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 40, 'total_tokens': 112, 'output_token_details': {'reasoning': 22}})

In [26]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# 1. Simple RAG Chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [28]:
## Basic retriever
retriever= vector_store.as_retriever(
    search_type="similarity",
    search_kwargs = {"k":3}
)

In [29]:
retriever

VectorStoreRetriever(tags=['AstraDBVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_astradb.vectorstores.AstraDBVectorStore object at 0x00000112656F2C10>, search_kwargs={'k': 3})

In [31]:
from langchain_core.runnables import (
    RunnablePassthrough, 
 
)
from langchain_core.output_parsers import StrOutputParser

In [30]:
from typing import List

# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i,doc in enumerate(docs):
        source = doc.metadata.get('source','Unknown')
        formatted.append(f"Document {i+1} (Source: {source}): \n{doc.page_content}")
    return "\n\n".join(formatted)


In [32]:
simple_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough() }
    | simple_prompt
    | llm
    |StrOutputParser()
)

In [33]:
simple_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])

In [35]:
### Conversational RAG Chain

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}"),
])

In [36]:
def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return (
        RunnablePassthrough.assign(
            context= lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [37]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [38]:
## Streaming RAG Chain
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)

print("Modern RAG chain created successfully!")
print("Available chains: ")
print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")

Modern RAG chain created successfully!
Available chains: 
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming


In [39]:
## Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)

    # 1. Simple RAG
    print("\n1. Simple RAG Chain: ")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    print("\n2. Streaming RAG: ")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [40]:
test_rag_chains("how many cash was robbers stole")

Question: how many cash was robbers stole

1. Simple RAG Chain: 
Answer: The robbers stole **$1 million in cash**.

2. Streaming RAG: 
Answer: The robbers stole **$1 million in cash**.


In [41]:
## Test with multiple questions
test_questions = [
    "tell the about the story and incident?",
    "calculate the loss after robbery",
    "who won"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)



Question: tell the about the story and incident?

1. Simple RAG Chain: 
Answer: Based on the provided context, the relevant story is:

Robbers broke into a city bank and stole $1 million in cash. This is the incident described in Document 1. The other two documents are unrelated personal tweets about a movie and a breakfast, and do not pertain to the incident.

2. Streaming RAG: 
Answer: Based on the provided context, the only incident described is a bank robbery:

- **Story/Incident:**  
  Robbers forced their way into the city bank and stole **$1 million in cash**.  

The other two documents are unrelated personal comments about a movie and a breakfast, and do not pertain to the story or incident.


Question: calculate the loss after robbery

1. Simple RAG Chain: 
Answer: The loss from the robbery is **$1 million**.

2. Streaming RAG: 
Answer: The loss from the robbery is **$1 million**.


Question: who won

1. Simple RAG Chain: 
Answer: I’m sorry, but the provided documents don’t 